In [0]:
%sql
CREATE MATERIALIZED VIEW small_projects.finops.mv_daily_cost
AS
SELECT
  usage_date,
  workspace_name,
  SUM(total_cost) AS total_cost,
  SUM(idle_cost) AS idle_cost,
  SUM(dbus) AS total_dbus
FROM small_projects.finops.gold_usage_cost
GROUP BY usage_date, workspace_name;

In [0]:
%sql
SELECT COUNT(*) AS row_count, ROUND(SUM(total_cost)) AS total_cost
FROM small_projects.finops.mv_daily_cost;

In [0]:
%sql
SELECT usage_date, workspace_name, ROUND(total_cost) AS total_cost
FROM small_projects.finops.mv_daily_cost
ORDER BY total_cost DESC
LIMIT 3;

In [0]:
%sql
REFRESH MATERIALIZED VIEW small_projects.finops.mv_daily_cost;

In [0]:
%sql

CREATE OR REPLACE TABLE small_projects.finops.demo_cost_by_team AS
SELECT tag_team, ROUND(SUM(total_cost)) AS total_cost
FROM small_projects.finops.gold_usage_cost
GROUP BY tag_team;

-- 1. Grants so `account users` can read the underlying table
GRANT USE CATALOG ON CATALOG small_projects TO `account users`;
GRANT USE SCHEMA ON SCHEMA small_projects.finops TO `account users`;
GRANT SELECT ON TABLE small_projects.finops.gold_usage_cost TO `account users`;

-- 2. Column mask: only members of the 'finance' group see the cost value
CREATE OR REPLACE FUNCTION small_projects.finops.mask_cost(cost DOUBLE)
RETURN IF(is_account_group_member('finance'), cost, NULL);

-- 3. Apply the mask to the total_cost column
ALTER TABLE small_projects.finops.demo_cost_by_team
  ALTER COLUMN total_cost SET MASK small_projects.finops.mask_cost;

-- 4. Test: total_cost shows values for 'finance' members, NULL for everyone else
SELECT *
FROM small_projects.finops.demo_cost_by_team
LIMIT 10;


In [0]:
%sql
CREATE OR REPLACE FUNCTION small_projects.finops.mask_cost(cost DOUBLE)
RETURNS DOUBLE
RETURN IF(is_account_group_member('account users'), cost, NULL);

SELECT * FROM small_projects.finops.demo_cost_by_team;

In [0]:
%sql
-- a) ABAC için yeni demo tablosu (üzerinde elle maske yok)
CREATE OR REPLACE TABLE small_projects.finops.demo_cost_abac AS
SELECT tag_team, ROUND(SUM(total_cost)) AS total_cost
FROM small_projects.finops.gold_usage_cost
GROUP BY tag_team;

-- b) Maliyet sütununu "confidential" olarak etiketle
ALTER TABLE small_projects.finops.demo_cost_abac
  ALTER COLUMN total_cost SET TAGS ('sensitivity' = 'confidential');

-- c) Değeri gizleyen fonksiyon
CREATE OR REPLACE FUNCTION small_projects.finops.hide_value(v DOUBLE)
RETURNS DOUBLE
RETURN CAST(NULL AS DOUBLE);

-- d) Tüm şema için TEK bir kural: "confidential" etiketli her sütunu maskele
CREATE OR REPLACE POLICY mask_confidential_columns
ON SCHEMA small_projects.finops
COLUMN MASK small_projects.finops.hide_value
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('sensitivity', 'confidential') AS c
ON COLUMN c;

-- e) Test
SELECT * FROM small_projects.finops.demo_cost_abac;

In [0]:
%sql
CREATE OR REPLACE TABLE small_projects.finops.demo_idle_abac AS
SELECT tag_team, ROUND(SUM(idle_cost)) AS idle_cost
FROM small_projects.finops.gold_usage_cost
GROUP BY tag_team;

ALTER TABLE small_projects.finops.demo_idle_abac
  ALTER COLUMN idle_cost SET TAGS ('sensitivity' = 'confidential');

SELECT * FROM small_projects.finops.demo_idle_abac;

In [0]:
%sql
ALTER TABLE small_projects.finops.demo_idle_abac
  ALTER COLUMN idle_cost UNSET TAGS ('sensitivity');

SELECT * FROM small_projects.finops.demo_idle_abac;